# Auditoria do planejamento da vigilância de Orlândia

## Síntese

O arquivo contém 2.371 registros; 1.949 não informam resultado de larvas. Na janela de 12/08 a 08/09/2026, há 14 ocorrências com larvas, 1.161 de imóveis fechados e 121 recusas. Estas são somas de registros, com possíveis revisitas. Não há contagem municipal de suspeitas disponível nesta análise.

## Contexto e método

Companheiro de auditoria do estudo `docs/plano_vigilancia_orlandia.md` e do painel `/sfa/entomologia`. Executar a partir da raiz do repositório ou deste diretório. Requer somente Python 3.10+. Não faz consulta externa nem grava dados de pacientes. Janelas inclusivas e adjacentes de 28 dias; nulos preservados. Tabelas exatas permitem conferir os números; a exploração visual está no painel.

### Premissas

Cada linha é um registro da fonte, não um imóvel único. Códigos de setor não comprovam equivalência territorial com a malha 2022. Zero somado pode ser parcial. O resultado não é IIP, incidência ou efeito causal de intervenção.

## Dados e procedência

In [1]:
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict
import json, hashlib
from pprint import pprint

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'services/data/entomologia/snapshot.json').exists())
source_path = root / 'services/data/entomologia/snapshot.json'
data = json.loads(source_path.read_text(encoding='utf-8'))
rows = data['records']
print('Arquivo: services/data/entomologia/snapshot.json')
print('SHA256 do snapshot:', hashlib.sha256(source_path.read_bytes()).hexdigest())
print('SHA256 do CSV recebido:', data['source']['sha256'])
print('Período:', data['source']['start'], 'a', data['source']['end'])
print('Censo:', data['source']['census_url'])

Arquivo: services/data/entomologia/snapshot.json
SHA256 do snapshot: b551841dea791f973baaa70eb0c96e52cce6b54ad6157ef117e9e61ccd8b487c
SHA256 do CSV recebido: 7e70d9de8d6064e954b54f48b36992ec38b00ff09cf139d8ddbf648052cac688
Período: 2026-01-05 a 2026-09-08
Censo: https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/malha_com_atributos/setores/shp/UF/SP/SP_setores_CD2022.zip


## Resultados

### Completude e contagens gerais

In [2]:
def known_sum(values, key):
    known = [r[key] for r in values if r[key] is not None]
    return sum(known) if known else None

def summary(values):
    filled = sum(r['positive'] is not None for r in values)
    return {'registros': len(values), 'setores': len({r['sector'] for r in values}),
            'trabalhados': known_sum(values, 'worked'), 'com_larvas': known_sum(values, 'positive'),
            'fechados': known_sum(values, 'closed'), 'recusas': known_sum(values, 'refused'),
            'informados': filled, 'sem_resultado': len(values)-filled,
            'preenchimento_pct': round(100*filled/len(values), 2) if values else None}

pprint(summary(rows), sort_dicts=False)
codes = {f['properties']['sector'] for f in data['census']['features']}
print('Registros sem código na malha:', sum(r['sector'] not in codes for r in rows))
for field in ['population', 'households', 'occupied_households']:
    print(field, sum(f['properties'][field] for f in data['census']['features']))

{'registros': 2371,
 'setores': 72,
 'trabalhados': 16507,
 'com_larvas': 215,
 'fechados': 13571,
 'recusas': 1207,
 'informados': 422,
 'sem_resultado': 1949,
 'preenchimento_pct': 17.8}
Registros sem código na malha: 43
population 38319
households 15334
occupied_households 13566


### Comparação entre janelas de mesma duração

In [3]:
end = date.fromisoformat(data['source']['end'])
days = 28
start = end - timedelta(days=days-1)
previous_end = start - timedelta(days=1)
previous_start = start - timedelta(days=days)
current = [r for r in rows if start.isoformat() <= r['date'] <= end.isoformat()]
previous = [r for r in rows if previous_start.isoformat() <= r['date'] <= previous_end.isoformat()]
print('Atual:', start, 'a', end)
pprint(summary(current), sort_dicts=False)
print('Anterior:', previous_start, 'a', previous_end)
pprint(summary(previous), sort_dicts=False)

Atual: 2026-08-12 a 2026-09-08
{'registros': 238,
 'setores': 38,
 'trabalhados': 1591,
 'com_larvas': 14,
 'fechados': 1161,
 'recusas': 121,
 'informados': 68,
 'sem_resultado': 170,
 'preenchimento_pct': 28.57}
Anterior: 2026-07-15 a 2026-08-11
{'registros': 250,
 'setores': 36,
 'trabalhados': 1529,
 'com_larvas': 15,
 'fechados': 1004,
 'recusas': 163,
 'informados': 69,
 'sem_resultado': 181,
 'preenchimento_pct': 27.6}


### Territórios citados no estudo

Valores conferidos por agregação independente do modelo JavaScript do painel. Nenhum nome de bairro foi inferido.

In [4]:
by_sector = defaultdict(list)
for row in current:
    by_sector[row['sector']].append(row)
for suffix in ['0017', '0003', '0045', '0018', '0076']:
    values = by_sector['35343020500' + suffix]
    s = summary(values)
    print('Setor', suffix, '| larvas', s['com_larvas'], '| A. aegypti', known_sum(values, 'aegypti'))
    print('fechados', s['fechados'], '| recusas', s['recusas'], '| sem resultado', s['sem_resultado'], '/', s['registros'])
    print('preenchimento (%)', s['preenchimento_pct'])
    print()

Setor 0017 | larvas 3 | A. aegypti 1
fechados 166 | recusas 8 | sem resultado 21 / 24
preenchimento (%) 12.5

Setor 0003 | larvas 2 | A. aegypti 1
fechados 0 | recusas 0 | sem resultado 0 / 3
preenchimento (%) 100.0

Setor 0045 | larvas 0 | A. aegypti 0
fechados 165 | recusas 25 | sem resultado 15 / 17
preenchimento (%) 11.76

Setor 0018 | larvas None | A. aegypti None
fechados 140 | recusas 14 | sem resultado 5 / 5
preenchimento (%) 0.0

Setor 0076 | larvas 0 | A. aegypti 0
fechados 104 | recusas 4 | sem resultado 27 / 28
preenchimento (%) 3.57



### Verificações de consistência

In [5]:
assert len(rows) == 2371
assert summary(rows)['sem_resultado'] == 1949
assert known_sum(rows, 'worked') == 16507
assert known_sum(rows, 'positive') == 215
assert summary(current)['registros'] == 238
assert summary(current)['com_larvas'] == 14
assert summary(current)['fechados'] == 1161
assert summary(current)['recusas'] == 121
assert summary(previous)['registros'] == 250
assert summary(previous)['com_larvas'] == 15
assert sum(len(values) for values in by_sector.values()) == len(current)
assert known_sum([{'positive': None}], 'positive') is None
assert known_sum([{'positive': 0}], 'positive') == 0
assert start - previous_start == timedelta(days=28)
assert not ({r['date'] for r in current} & {r['date'] for r in previous})
print('Verificações concluídas; nenhuma divergência nas contagens citadas.')

Verificações concluídas; nenhuma divergência nas contagens citadas.


## Interpretação

O setor 0017 reúne sinais para conferir focos e acesso; o setor 0045 se destaca por fechados e recusas. A incompletude impede tratar resultados ausentes como ausência de infestação. A situação atual dos imóveis e as intervenções concluídas não estão na fonte. A comparação entre 14 e 15 ocorrências não permite concluir que a transmissão caiu.

Fonte epidemiológica e cobertura da coorte continuam pendentes. Consultar o estudo para programas, organização das tabelas e indicadores de acompanhamento. Orientação de referência: https://www.gov.br/saude/pt-br/assuntos/saude-de-a-a-z/a/aedes-aegypti/vigilancia-entomologica